In [1]:
import os
import numpy as np
import pandas as pd
import diptest

from sklearn.mixture import GaussianMixture


INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.csv"

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_CSV}"
    )

df = pd.read_csv(INPUT_CSV)

print("GBM MULTIMODALITY ANALYSIS")
print("=" * 70)

print(
    f"Loaded membrane components : {len(df)}"
)

print(
    f"Patients: {df['patient_id'].nunique()}"
)

def fit_gmm_models(values):

    X = values.reshape(-1,1)

    models = {}
    bic = {}
    aic = {}

    for n in [1,2,3]:

        model = GaussianMixture(
            n_components=n,
            random_state=42,
            n_init=20
        )

        model.fit(X)

        models[n] = model
        bic[n] = model.bic(X)
        aic[n] = model.aic(X)


    return models, bic, aic

results = []

for patient_id, group in df.groupby("patient_id"):

    print("\n" + "-"*70)

    thickness = (
        group["median_thickness_nm"]
        .dropna()
        .to_numpy()
    )

    n_samples = len(thickness)

    print(
        f"{'Patient':<25}: {patient_id}"
    )

    print(
        f"{'Membrane components':<25}: {n_samples}"
    )

    if n_samples < 5:

        print(
            "Skipped - too few samples"
        )
        continue

    # Hartigan Dip Test
    dip_statistic, dip_pvalue = diptest.diptest(
        thickness
    )

    if dip_pvalue < 0.05:

        dip_classification = "Multimodal"

    else:

        dip_classification = "Unimodal"


    # GMM analysis
    models, bic, aic = fit_gmm_models(
        thickness
    )

    best_components = min(
        bic,
        key=bic.get
    )

    best_model = models[best_components]
    peaks = best_model.means_.flatten()
    weights = best_model.weights_.flatten()
    order = np.argsort(peaks)
    peaks = peaks[order]
    weights = weights[order]

    peaks = np.pad(
        peaks,
        (0,3-len(peaks)),
        constant_values=np.nan
    )

    weights = np.pad(
        weights,
        (0,3-len(weights)),
        constant_values=np.nan
    )

    if best_components >= 2:

        peak_distance = peaks[1]-peaks[0]

    else:

        peak_distance = np.nan


    if dip_classification == "Unimodal":

        if best_components == 1:

            interpretation = (
                "Unimodal distribution"
            )

        else:

            interpretation = (
                "Unimodal distribution with multiple Gaussian components"
            )


    else:

        interpretation = (
            "Multimodal distribution"
        )


    if dip_pvalue < 0.01:

        evidence = "Strong"

    elif dip_pvalue < 0.05:

        evidence = "Moderate"

    else:

        evidence = "Not Detected"


    print(
        f"{'Dip statistic':<25}: {dip_statistic:.5f}"
    )

    print(
        f"{'Dip p-value':<25}: {dip_pvalue:.5f}"
    )

    print(
        f"{'Dip classification':<25}: {dip_classification}"
    )

    print(
        f"{'Best GMM':<25}: {best_components} component(s)"
    )

    print(
        f"{'BIC':<25}: {bic[best_components]:.2f}"
    )

    print(
        f"{'AIC':<25}: {aic[best_components]:.2f}"
    )

    print(
        f"{'Peak positions':<25}: {np.round(peaks,2)}"
    )

    print(
        f"{'Peak weights':<25}: {np.round(weights,3)}"
    )


    if not np.isnan(peak_distance):

        print(
            f"{'Peak separation':<25}: {peak_distance:.2f} nm"
        )

    else:

        print(
            f"{'Peak separation':<25}: N/A"
        )


    print(
        f"{'GMM interpretation':<25}: {interpretation}"
    )


    print(
        f"{'Statistical evidence':<25}: {evidence}"
    )

    results.append({

        "Patient_ID": patient_id,
        "Number_of_membranes": n_samples,
        "Dip_statistic": dip_statistic,
        "Dip_p_value": dip_pvalue,
        "Dip_classification": dip_classification,
        "Best_GMM_components": best_components,
        "BIC": bic[best_components],
        "AIC": aic[best_components],
        "Peak_1_nm": peaks[0],
        "Peak_2_nm": peaks[1],
        "Peak_3_nm": peaks[2],
        "Weight_1": weights[0],
        "Weight_2": weights[1],
        "Weight_3": weights[2],
        "Peak_separation_nm": peak_distance,
        "GMM_interpretation": interpretation,
        "Statistical_evidence": evidence

    })


summary_df = pd.DataFrame(results)

summary_df = summary_df.sort_values(
    by=[
        "Dip_p_value",
        "Best_GMM_components",
        "BIC"
    ],
    ascending=[
        True,
        False,
        True
    ]
).reset_index(drop=True)

summary_df.insert(
    0,
    "Rank",
    np.arange(1,len(summary_df)+1)
)

summary_df.to_csv(
    OUTPUT_CSV,
    index=False
)

print("\n")
print("="*70)
print("FINAL SUMMARY")
print("="*70)

print(
    summary_df[
        [
            "Rank",
            "Patient_ID",
            "Dip_p_value",
            "Dip_classification",
            "Best_GMM_components",
            "GMM_interpretation",
            "Statistical_evidence"
        ]
    ].to_string(index=False)
)

print("\n")
print("Results saved:")
print(OUTPUT_CSV)

GBM MULTIMODALITY ANALYSIS
Loaded membrane components : 394
Patients: 11

----------------------------------------------------------------------
Patient                  : 01-24
Membrane components      : 19
Dip statistic            : 0.06017
Dip p-value              : 0.85855
Dip classification       : Unimodal
Best GMM                 : 3 component(s)
BIC                      : 204.34
AIC                      : 196.78
Peak positions           : [170.85 377.78 480.62]
Peak weights             : [0.842 0.053 0.105]
Peak separation          : 206.94 nm
GMM interpretation       : Unimodal distribution with multiple Gaussian components
Statistical evidence     : Not Detected

----------------------------------------------------------------------
Patient                  : 02-24
Membrane components      : 41
Dip statistic            : 0.03233
Dip p-value              : 0.99183
Dip classification       : Unimodal
Best GMM                 : 3 component(s)
BIC                      : 477.17
AI